# inference-bench on Colab

**Runtime > Change runtime type > T4 GPU** first. The full run takes about 30 to 45 minutes on a free T4.

Results are written straight to **Google Drive** (`My Drive/inference-bench-results`), so they survive a disconnect.
Approve the Drive popup when the setup cell asks. Keep this tab open while it runs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Safe to rerun: always clones into /content/inference-bench, never a nested copy.
%cd /content
REPO = "https://github.com/Shri2007/inference-bench.git"
!test -d inference-bench || git clone -q $REPO
%cd /content/inference-bench
!git pull -q

# Save results straight to Google Drive so they survive a disconnect or a recycled machine.
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/inference-bench-results
!rm -rf results && ln -s /content/drive/MyDrive/inference-bench-results results
!ls -la results

Installing vLLM takes a few minutes. Ignore pip warnings about torch versions.

In [ ]:
!pip install -q vllm matplotlib

## 1. Baseline (no speculation)

In [ ]:
!python bench.py --spec none

## 2. N-gram speculation (prompt lookup)
Guesses the next tokens by finding the last few tokens earlier in the prompt. No extra model.

In [ ]:
!python bench.py --spec ngram --k 4

## 3. Draft-model speculation
A 0.5B model drafts tokens and the 1.5B model checks them. If your vLLM version doesn't support this, the run records the error and the report skips it.

In [ ]:
!python bench.py --spec draft --k 4

## 4. Report

In [ ]:
!python report.py
from IPython.display import Image, display, Markdown
display(Image("results/throughput.png")); display(Image("results/speedup.png"))

## 4b. Does speculation change the text?
Compares baseline against itself across batch sizes as a control.

In [ ]:
!python check_match.py

## 5. Get the results
Everything is already in Google Drive at `My Drive/inference-bench-results`. Download that folder from drive.google.com.